In [18]:
from datasets import load_from_disk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz
import pickle

In [19]:
likes = load_from_disk('/Users/layvvs/Desktop/HSE/Studying/year-project/hse-ai-year-project-2025/checkpoint-5-bogdan-egor/bogdan/dataset-downloading/yambda_likes')
listens = load_from_disk('/Users/layvvs/Desktop/HSE/Studying/year-project/hse-ai-year-project-2025/checkpoint-5-bogdan-egor/bogdan/dataset-downloading/yambda_listens')

likes_table = likes.data.table
listens_table = listens.data.table

likes_df: pd.DataFrame = likes_table.to_pandas()
listens_df: pd.DataFrame = listens_table.to_pandas()

# likes_df: pd.DataFrame = pd.DataFrame.from_arrow(likes_table)
# listens_df: pd.DataFrame = pd.DataFrame.from_arrow(listens_table)

likes_df['event_type'] = 'like'
listens_df['event_type'] = 'listen'

yambda_df = pd.concat([likes_df, listens_df], ignore_index=True)

yambda_df = yambda_df.sort_values('timestamp')

yambda_df

,uid,timestamp,item_id,is_organic,event_type,played_ratio_pct,track_length_seconds
22879986,468300,0,7400764,0,listen,100.0,225.0
17033976,347600,5,3415205,0,listen,100.0,250.0
44773176,942900,10,6728180,0,listen,1.0,270.0
1388942,12700,15,8932363,0,listen,100.0,245.0
12164540,243500,15,5283544,1,listen,100.0,195.0
...,...,...,...,...,...,...,...
30815427,645300,25999995,1172812,1,listen,100.0,120.0
9620970,190700,25999995,875069,0,listen,0.0,235.0
7523389,144700,26000000,6281650,0,listen,6.0,205.0
42998457,903600,26000000,8370890,1,listen,6.0,220.0


Оставим только те uid и item_id, что есть в матрице интеракций

In [20]:
with open('../data/mappings_item_filters_user_filters.pkl', 'rb') as f:
    mappings = pickle.load(f)

uid2idx = mappings['user2id']
iid2idx = mappings['item2id']

valid_uids = set(uid2idx.keys())
valid_iids = set(iid2idx.keys())

yambda_df_filtered = yambda_df[
    yambda_df['uid'].isin(valid_uids) & 
    yambda_df['item_id'].isin(valid_iids)
]

In [21]:
yambda_df_filtered

,uid,timestamp,item_id,is_organic,event_type,played_ratio_pct,track_length_seconds
12164540,243500,15,5283544,1,listen,100.0,195.0
46995080,992600,20,140621,0,listen,100.0,270.0
36969365,771700,20,6099388,0,listen,100.0,215.0
30204741,631600,20,1652612,0,listen,100.0,155.0
37768322,789000,25,3001214,1,listen,100.0,210.0
...,...,...,...,...,...,...,...
42851737,899900,25999995,4027449,1,listen,100.0,240.0
26737229,554500,25999995,1011428,1,listen,3.0,115.0
30815427,645300,25999995,1172812,1,listen,100.0,120.0
42998457,903600,26000000,8370890,1,listen,6.0,220.0


In [22]:
with open('../data/mappings_item_filters_user_filters.pkl', 'rb') as f:
    mappings = pickle.load(f)

user2id = mappings['user2id']
item2id = mappings['item2id']
id2item = mappings['id2item']
id2user = mappings['id2user']

user_item_matrix = load_npz('../data/user_item_matrix_item_filters_user_filters.npz')

with open('../data/test_true_item_filters_user_filters.pkl', 'rb') as f:
    test_true = pickle.load(f)

weak_ticks = (7 * 24 * 3600) // 5
year_ticks = (365 * 24 * 3600) // 5

max_time = yambda_df['timestamp'].max()
test_start = max_time - weak_ticks
train_start = test_start - year_ticks

train_events = yambda_df_filtered[
    (yambda_df_filtered['timestamp'] >= train_start) &
    (yambda_df_filtered['timestamp'] < test_start)
].copy()

print(f'train events: {len(train_events):,}')
print(f'users in matrix: {user_item_matrix.shape[0]:,}, items: {user_item_matrix.shape[1]:,}')
print(f'test users: {len(test_true):,}')

train events: 10,496,309
users in matrix: 6,477, items: 200,250
test users: 4,429


# User Features

In [23]:
user_features = train_events.groupby('uid').agg(
    user_n_events=('item_id', 'size'),
    n_unique_items=('item_id', 'nunique'),
    user_n_likes=('event_type', lambda s: (s == 'like').sum()),
    n_listens=('event_type', lambda s: (s == 'listen').sum()),
    user_avg_played_ratio=('played_ratio_pct', 'mean'),
    user_organic_ratio=('is_organic', 'mean'),
).reset_index()

user_features['user_like_rate'] = user_features['user_n_likes'] / user_features['user_n_events']

user_entropy = pd.read_csv('../data/users-entropy.csv', index_col=0)
user_features = user_features.merge(
    user_entropy[['uid', 'entropy', 'entropy_norm', 'n_items']],
    on='uid',
    how='left',
)

matrix_n_positive = np.array((user_item_matrix > 0).sum(axis=1)).ravel()
matrix_total_weight = np.array(user_item_matrix.sum(axis=1)).ravel()

matrix_user_stats = pd.DataFrame({
    'uid': [id2user[i] for i in range(len(matrix_n_positive))],
    'matrix_n_positive': matrix_n_positive,
    'matrix_total_weight': matrix_total_weight,
})
user_features = user_features.merge(matrix_user_stats, on='uid', how='left')

user_features.head()

,uid,user_n_events,n_unique_items,user_n_likes,n_listens,user_avg_played_ratio,user_organic_ratio,user_like_rate,entropy,entropy_norm,n_items,matrix_n_positive,matrix_total_weight
0,100,1032,488,6,1026,84.291423,0.327519,0.005814,5.907142,0.953312,491.0,346,346.0
1,600,4042,936,0,4042,99.541811,0.000000,0.000000,6.611231,0.964382,949.0,919,919.0
2,700,2282,1090,0,2282,63.384750,0.351884,0.000000,6.730655,0.953635,1162.0,430,430.0
3,800,2488,1269,19,2469,68.889429,0.528135,0.007637,6.874831,0.956679,1321.0,603,603.0
4,900,289,121,20,269,48.230483,0.858131,0.069204,4.399950,0.906827,128.0,21,21.0


# Item Features

In [24]:
item_features = train_events.groupby('item_id').agg(
    item_n_events=('uid', 'size'),
    n_unique_users=('uid', 'nunique'),
    item_n_likes=('event_type', lambda s: (s == 'like').sum()),
    item_avg_played_ratio=('played_ratio_pct', 'mean'),
    avg_track_length=('track_length_seconds', 'mean'),
    item_organic_ratio=('is_organic', 'mean'),
).reset_index()

item_features['item_like_rate'] = item_features['item_n_likes'] / item_features['item_n_events']

matrix_item_pop = np.array(user_item_matrix.sum(axis=0)).ravel()
matrix_item_stats = pd.DataFrame({
    'item_id': [id2item[i] for i in range(len(matrix_item_pop))],
    'matrix_popularity': matrix_item_pop,
})
item_features = item_features.merge(matrix_item_stats, on='item_id', how='left')

item_features.head()

,item_id,item_n_events,n_unique_users,item_n_likes,item_avg_played_ratio,avg_track_length,item_organic_ratio,item_like_rate,matrix_popularity
0,50,44,26,1,77.325581,215.0,0.113636,0.022727,18.0
1,175,7,4,0,76.571429,180.0,0.000000,0.000000,2.0
2,195,3,3,0,99.000000,190.0,0.333333,0.000000,3.0
3,198,16,11,1,63.000000,180.0,0.062500,0.062500,6.0
4,206,137,77,2,46.925926,225.0,0.386861,0.014599,22.0


# User-Item Features

In [25]:
ui_stats = train_events.groupby(['uid', 'item_id']).agg(
    ui_n_events=('timestamp', 'size'),
    mean_played_ratio=('played_ratio_pct', 'mean'),
    has_like=('event_type', lambda s: (s == 'like').any()),
    last_timestamp=('timestamp', 'max'),
    organic_ratio_ui=('is_organic', 'mean'),
).reset_index()

ui_stats['days_before_test'] = (test_start - ui_stats['last_timestamp']) * 5 / (24 * 3600)

coo = user_item_matrix.tocoo()
ui_matrix_weight = pd.DataFrame({
    'uid': [id2user[r] for r in coo.row],
    'item_id': [id2item[c] for c in coo.col],
    'interaction_weight': coo.data,
})

ui_features = ui_stats.merge(ui_matrix_weight, on=['uid', 'item_id'], how='left')
ui_features.head()

,uid,item_id,ui_n_events,mean_played_ratio,has_like,last_timestamp,organic_ratio_ui,days_before_test,interaction_weight
0,100,6732,1,100.000000,False,24750055,0.0,65.334780,1.0
1,100,19712,1,24.000000,False,24502950,0.0,79.634838,0.0
2,100,27230,2,100.000000,False,24688620,0.0,68.890046,1.0
3,100,88111,4,68.250000,False,25619825,0.5,15.000868,0.0
4,100,102185,3,41.666667,False,24599475,0.0,74.048900,0.0


# Reranker Dataset

Собираем выборку для реранкера:
1. ELSA даёт top-K кандидатов на каждого test-пользователя
2. label = 1, если айтем есть в `test_true`, иначе 0
3. к каждой паре (uid, item_id) джойним user / item / user-item фичи
4. делим пользователей на train / test (80 / 20)

In [26]:
import sys
sys.path.append('../check6')
from models import ELSA

CANDIDATE_K = 100

with open('elsa_training_item_filters_user_filters.pkl', 'rb') as f:
    elsa = pickle.load(f)

rows = []
for uid, true_items in test_true.items():
    candidates = elsa.predict(uid, user_item_matrix, user2id, id2item, k=CANDIDATE_K)
    for rank, item_id in enumerate(candidates):
        rows.append({
            'uid': uid,
            'item_id': item_id,
            'elsa_rank': rank,
            'label': int(item_id in true_items),
        })

candidates_df = pd.DataFrame(rows)
print(f'candidates: {len(candidates_df):,}, positive rate: {candidates_df["label"].mean():.4f}')
candidates_df.head()

candidates: 442,900, positive rate: 0.0189


,uid,item_id,elsa_rank,label
0,100,7092180,0,1
1,100,2981607,1,0
2,100,5013240,2,0
3,100,1938919,3,0
4,100,4398449,4,0


In [29]:
USER_FEATURE_COLS = [
    'user_n_events', 'n_unique_items', 'user_n_likes', 'n_listens', 'user_like_rate',
    'user_avg_played_ratio', 'user_organic_ratio', 'entropy', 'entropy_norm', 'n_items',
    'matrix_n_positive', 'matrix_total_weight',
]
ITEM_FEATURE_COLS = [
    'item_n_events', 'n_unique_users', 'item_n_likes', 'item_like_rate',
    'item_avg_played_ratio', 'avg_track_length', 'item_organic_ratio', 'matrix_popularity',
]
UI_FEATURE_COLS = [
    'ui_n_events', 'mean_played_ratio', 'has_like', 'days_before_test',
    'organic_ratio_ui', 'interaction_weight',
]
FEATURE_COLS = ['elsa_rank'] + USER_FEATURE_COLS + ITEM_FEATURE_COLS + UI_FEATURE_COLS


def build_reranker_df(candidates: pd.DataFrame) -> pd.DataFrame:
    df = (
        candidates
        .merge(user_features[['uid'] + USER_FEATURE_COLS], on='uid', how='left')
        .merge(item_features[['item_id'] + ITEM_FEATURE_COLS], on='item_id', how='left')
        .merge(
            ui_features[['uid', 'item_id', 'ui_n_events', 'mean_played_ratio', 'has_like',
                         'days_before_test', 'organic_ratio_ui', 'interaction_weight']],
            on=['uid', 'item_id'],
            how='left',
        )
    )
    df['has_like'] = df['has_like'].fillna(False).astype(int)
    df['interaction_weight'] = df['interaction_weight'].fillna(0.0)
    df['ui_n_events'] = df['ui_n_events'].fillna(0)
    df['days_before_test'] = df['days_before_test'].fillna(-1)
    return df


test_uids = np.array(list(test_true.keys()))
rng = np.random.default_rng(42)
rng.shuffle(test_uids)

split_idx = int(len(test_uids) * 0.8)
train_uids = set(test_uids[:split_idx])
test_uids_holdout = set(test_uids[split_idx:])

candidates_train = candidates_df[candidates_df['uid'].isin(train_uids)]
candidates_test = candidates_df[candidates_df['uid'].isin(test_uids_holdout)]

reranker_train_df = build_reranker_df(candidates_train)
reranker_test_df = build_reranker_df(candidates_test)

print(f'train users: {len(train_uids):,}, test users: {len(test_uids_holdout):,}')
print(f'reranker_train_df: {reranker_train_df.shape}, positive rate: {reranker_train_df["label"].mean():.4f}')
print(f'reranker_test_df:  {reranker_test_df.shape}, positive rate: {reranker_test_df["label"].mean():.4f}')
print(f'features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
reranker_train_df.head()

/var/folders/3t/gyxj9s512yx6klv5c576ykyc0000gn/T/ipykernel_28217/595611258.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['has_like'] = df['has_like'].fillna(False).astype(int)


train users: 3,543, test users: 886
reranker_train_df: (354300, 30), positive rate: 0.0194
reranker_test_df:  (88600, 30), positive rate: 0.0168
features (27): ['elsa_rank', 'user_n_events', 'n_unique_items', 'user_n_likes', 'n_listens', 'user_like_rate', 'user_avg_played_ratio', 'user_organic_ratio', 'entropy', 'entropy_norm', 'n_items', 'matrix_n_positive', 'matrix_total_weight', 'item_n_events', 'n_unique_users', 'item_n_likes', 'item_like_rate', 'item_avg_played_ratio', 'avg_track_length', 'item_organic_ratio', 'matrix_popularity', 'ui_n_events', 'mean_played_ratio', 'has_like', 'days_before_test', 'organic_ratio_ui', 'interaction_weight']


/var/folders/3t/gyxj9s512yx6klv5c576ykyc0000gn/T/ipykernel_28217/595611258.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['has_like'] = df['has_like'].fillna(False).astype(int)


,uid,item_id,elsa_rank,label,user_n_events,n_unique_items,user_n_likes,n_listens,user_like_rate,user_avg_played_ratio,...,item_avg_played_ratio,avg_track_length,item_organic_ratio,matrix_popularity,ui_n_events,mean_played_ratio,has_like,days_before_test,organic_ratio_ui,interaction_weight
0,100,7092180,0,1,1032,488,6,1026,0.005814,84.291423,...,64.037598,180.0,0.656442,209.0,0.0,NaN,0,-1.000000,NaN,0.0
1,100,2981607,1,0,1032,488,6,1026,0.005814,84.291423,...,59.659229,180.0,0.627412,181.0,2.0,51.000000,0,68.973669,0.0,0.0
2,100,5013240,2,0,1032,488,6,1026,0.005814,84.291423,...,58.494415,185.0,0.689906,176.0,0.0,NaN,0,-1.000000,NaN,0.0
3,100,1938919,3,0,1032,488,6,1026,0.005814,84.291423,...,58.692202,205.0,0.788070,203.0,0.0,NaN,0,-1.000000,NaN,0.0
4,100,4398449,4,0,1032,488,6,1026,0.005814,84.291423,...,63.838284,195.0,0.620521,130.0,3.0,67.333333,0,68.898148,0.0,0.0


In [30]:
reranker_test_df.head()

,uid,item_id,elsa_rank,label,user_n_events,n_unique_items,user_n_likes,n_listens,user_like_rate,user_avg_played_ratio,...,item_avg_played_ratio,avg_track_length,item_organic_ratio,matrix_popularity,ui_n_events,mean_played_ratio,has_like,days_before_test,organic_ratio_ui,interaction_weight
0,1100,1489967,0,0,2326,1299,62,2264,0.026655,87.360866,...,81.744711,150.0,0.175316,264.0,3.0,66.666667,0,176.959201,0.0,0.0
1,1100,5002269,1,0,2326,1299,62,2264,0.026655,87.360866,...,90.794231,135.0,0.015385,248.0,3.0,70.000000,0,136.439525,0.0,0.0
2,1100,169228,2,0,2326,1299,62,2264,0.026655,87.360866,...,88.724760,125.0,0.098086,322.0,3.0,78.333333,0,136.522569,0.0,0.0
3,1100,1014812,3,0,2326,1299,62,2264,0.026655,87.360866,...,94.730159,125.0,0.007937,180.0,0.0,NaN,0,-1.000000,NaN,0.0
4,1100,5752756,4,0,2326,1299,62,2264,0.026655,87.360866,...,93.247396,135.0,0.036458,183.0,0.0,NaN,0,-1.000000,NaN,0.0


In [31]:
reranker_train_df.to_csv('reranker-train.csv')
reranker_test_df.to_csv('reranker-test.csv')